# RAG — Retrieval Augmented Generation

## Jak sprawić, żeby LLM "wiedział" o rzeczach, których nie ma w jego danych treningowych?

### Skąd przychodzimy?

W warsztacie **Function Calling** zbudowaliśmy `search_presidents()` — wyszukiwarkę po prezydentach Polski.
Działała na dopasowaniu słów kluczowych. Widzieliśmy, że pytania typu *"kto zbierał monety"* lub *"związkowiec"* ją łamią.

Dziś zobaczymy **trzy podejścia** do tego problemu:

| # | Metoda | Jak działa | Skaluje się? |
|---|---|---|---|
| 1 | **Substring search** | `if query in text` | ✓ ale nie rozumie języka |
| 2 | **Context stuffing** | Cały tekst → do LLM-a | ✗ limit tokenów |
| 3 | **RAG (embeddingi)** | Wyszukaj top-K → podaj LLM-owi | ✓ + rozumie język |

### Plan warsztatu

1. **Pokażemy problem**: substring search nie rozumie synonimów
2. **Context stuffing**: wpychamy cały dokument do LLM-a — działa, ale nie skaluje się
3. **Embeddingi**: zamienimy tekst na wektory i pokażemy wyszukiwanie semantyczne
4. **RAG**: skleimy wyszukiwanie z LLM-em — i zobaczymy różnicę!

## 1. Konfiguracja środowiska

**Google Colab / MyBinder?** Odkomentuj i uruchom poniższą komórkę — pobierze potrzebne pliki.

In [ ]:
# import os
# _repo = 'https://raw.githubusercontent.com/NXTRSS/MachineLearningCourse/main'
# for f in ['utils.py', 'prezydenci_polski_biografie.txt', 'prezydenci_polski.md']:
#     if not os.path.exists(f):
#         !curl -sO {_repo}/{f}
# print('Pliki pobrane!')

In [1]:
from utils import ensure_package

ensure_package("openai")
ensure_package("sentence-transformers", "sentence_transformers")
ensure_package("numpy")

import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from IPython.display import display, Markdown
import textwrap

print("Pakiety załadowane!")

Pakiety załadowane!


## 2. Połączenie z LLM-em

Poniższa komórka automatycznie szuka działającego LLM-a. Zanim ją uruchomisz:

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:12px; border-radius:4px;">

**Upewnij się, że LLM jest uruchomiony!**

- **LM Studio** (macOS / Windows / Linux) — otwórz aplikację → załaduj model → zakładka *Local Server* → zielony przycisk *Start Server*
- **Ollama** — otwórz terminal i wpisz `ollama serve`
  - *macOS:* jeśli zainstalowałeś przez `.dmg`, Ollama zwykle startuje automatycznie (ikonka w pasku menu)
  - *Windows:* Ollama działa jako usługa po instalacji — jeśli nie, uruchom ręcznie z menu Start
  - *Linux:* `ollama serve` w terminalu (lub `systemctl start ollama` jeśli zainstalowano przez `curl`)

Szczegóły instalacji: patrz `setup_local_llm.ipynb` lub `docs/LOKALNE_LLM.md`
</div>

**Na zajęciach pracujemy na:** `gemma4:e4b` — szybki, lekki model. Pobierz go w LM Studio!

W domu możesz eksperymentować z większymi modelami: `qwen3.6`, `qwen3.5:9b+`, `gemma4:12b`
<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; margin-top:10px; border-radius:4px;">

**Google Colab?** Nie masz lokalnego LLM-a — wpisz adres serwera prowadzącego w `LECTURER_SERVER` poniżej.
Prowadzący poda URL tunnelu na zajęciach (np. `https://abc123.ngrok-free.app`).
</div>

In [2]:
from utils import connect_llm, extract_reasoning, print_reasoning, clean_content
from utils import setup_auth_client

# Jeśli nie masz lokalnego LLM-a, wpisz adres serwera prowadzącego (podany na zajęciach):
LECTURER_SERVER = "http://ADRES_SERWERA:PORT"  # ← prowadzący poda na zajęciach

client, instructor_client, MODEL_NAME = connect_llm(
    lecturer_server=LECTURER_SERVER,
    model="gemma-4-e4b",  # ← odkomentuj, by wybrać konkretny model (np. "qwen", "llama")
    # backend="lmstudio",          # ← wymuś backend: "lmstudio", "ollama", "lecturer"
    # backend=["lmstudio", "ollama"],  # ← lub lista — próbuj w kolejności
    ports=[4141],              # ← dodatkowe porty LM Studio na localhost
)

# Serwer prowadzącego? → poprosi o imię (i hasło jeśli wymagane)
client, instructor_client = setup_auth_client(client, instructor_client, MODEL_NAME)

# connect_llm zwraca DWA klienty:
#   client            → do function calling (LLM wybiera narzędzia)
#   instructor_client → do structured output (LLM odpowiada w formacie Pydantic)

if client:
    print(f"\nKlient LLM gotowy!  Model: {MODEL_NAME}")
    print(f"Instructor:         {'tak' if instructor_client else 'nie'}")
    print()
    print("Mamy DWA klienty:")
    print("  client            → do function calling (LLM wybiera narzędzia)")
    print("  instructor_client → do structured output (LLM odpowiada w formacie Pydantic)")


Szukam LM Studio (port 1234)...
  Uruchamiam LM Studio (`lms server start`)...
Szukam LM Studio (port 4141)...
✓ LM Studio (port 4141)! Model: gemma-4-e4b-it-mlx

Klient LLM gotowy!  Model: gemma-4-e4b-it-mlx
Instructor:         nie

Mamy DWA klienty:
  client            → do function calling (LLM wybiera narzędzia)
  instructor_client → do structured output (LLM odpowiada w formacie Pydantic)


## 3. Problem: substring search nie rozumie języka

W Function Calling zbudowaliśmy `search_presidents()` — wyszukiwarkę po słowach kluczowych.
Zobaczmy na czym polega jej ograniczenie:

In [3]:
from pathlib import Path

def load_presidents():
    """Ładuje dane o prezydentach z pliku prezydenci_polski.md"""
    md_path = Path("prezydenci_polski.md")
    if not md_path.exists():
        return []
    text = md_path.read_text(encoding="utf-8")
    presidents = []
    current = {}
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("### ") and not line.startswith("####"):
            if current:
                presidents.append(current)
            current = {"imię": line[4:].strip()}
        elif line.startswith("- **") and ":**" in line:
            key = line.split("**")[1].rstrip(":")
            value = line.split(":** ", 1)[1] if ":** " in line else ""
            current[key.lower()] = value
    if current:
        presidents.append(current)
    return presidents

PREZYDENCI = load_presidents()

def search_presidents(query: str) -> str:
    """
    Najprostsza wyszukiwarka — WSZYSTKIE słowa z zapytania muszą
    wystąpić w tekście o prezydencie. Żadnego scoringu, żadnej
    heurystyki — czysty substring matching.
    (W FC mieliśmy ulepszoną wersję ze scoringiem, tutaj celowo
    wracamy do bazowej żeby pokazać ograniczenia.)
    """
    if not PREZYDENCI:
        return "Brak danych — nie znaleziono pliku prezydenci_polski.md"
    query_lower = query.lower()
    words = query_lower.split()
    wyniki = []
    for p in PREZYDENCI:
        all_text = " ".join(f"{k} {v}" for k, v in p.items()).lower()
        if all(w in all_text for w in words):
            lines = [f"### {p.get('imię', '?')}"]
            for key, val in p.items():
                if key != 'imię':
                    lines.append(f"  - {key}: {val}")
            wyniki.append("\n".join(lines))
    if wyniki:
        return "\n\n".join(wyniki)
    return f"Nie znaleziono wyników dla zapytania: {query}"

print(f"Załadowano {len(PREZYDENCI)} prezydentów z prezydenci_polski.md")

Załadowano 7 prezydentów z prezydenci_polski.md


In [4]:
# Zapytania na których substring search ZAWODZI:
test_queries = [
    ("Nobel",                          "odmiana: 'Nobel' ≠ 'Nobla' w tekście"),
    ("kto zbierał monety",            "semantyczne — brak tych słów w tekście"),
    ("najdłużej rządził",             "synonim: 'rządził' ≠ 'kadencja'"),
    ("związkowiec",                    "w tekście 'NSZZ Solidarność', nie 'związkowiec'"),
    ("wojskowy",                       "w tekście 'Wyższa Szkoła Piechoty', nie 'wojskowy'"),
    ("Duda",                             "to akurat znajdzie — szuka dokładnego nazwiska"),
]

print("Substring search — test:\n")
for query, why in test_queries:
    result = search_presidents(query)
    found = "Nie znaleziono" not in result
    status = "ZNALAZŁ ✓" if found else "NIE ZNALAZŁ ✗"
    print(f'  "{query}"')
    print(f"    → {status}  ({why})")
    if found:
        first_line = result.split("\n")[0]
        print(f"    → {first_line}")
    print()

print("Substring search działa tylko gdy pytamy DOKŁADNYMI słowami z tekstu.")
print("A gdyby LLM mógł sam przeczytać cały dokument...?")

Substring search — test:

  "Nobel"
    → NIE ZNALAZŁ ✗  (odmiana: 'Nobel' ≠ 'Nobla' w tekście)

  "kto zbierał monety"
    → NIE ZNALAZŁ ✗  (semantyczne — brak tych słów w tekście)

  "najdłużej rządził"
    → NIE ZNALAZŁ ✗  (synonim: 'rządził' ≠ 'kadencja')

  "związkowiec"
    → NIE ZNALAZŁ ✗  (w tekście 'NSZZ Solidarność', nie 'związkowiec')

  "wojskowy"
    → NIE ZNALAZŁ ✗  (w tekście 'Wyższa Szkoła Piechoty', nie 'wojskowy')

  "Duda"
    → ZNALAZŁ ✓  (to akurat znajdzie — szuka dokładnego nazwiska)
    → ### Andrzej Duda

Substring search działa tylko gdy pytamy DOKŁADNYMI słowami z tekstu.
A gdyby LLM mógł sam przeczytać cały dokument...?


## 4. Context stuffing — wrzuć cały dokument do LLM-a

Najprostsze rozwiązanie: zamiast szukać, **dajmy LLM-owi cały tekst** i niech sam znajdzie odpowiedź.

```
Użytkownik: "Kto zbierał monety?"
    ↓
Prompt: "Oto dane o prezydentach: [CAŁY TEKST]. Odpowiedz na pytanie: ..."
    ↓
LLM: "Andrzej Duda — w tekście jest mowa o numizmatyce"
```

To działa! Ale jest haczyk — **co jeśli dokument ma 1000 stron?** LLM ma limit tokenów (context window).
Zobaczmy jak to wygląda na naszym małym zbiorze danych:

In [33]:
# === Context stuffing — cały dokument w prompcie ===
import time

_PRESIDENTS_RAW = Path("prezydenci_polski.md").read_text(encoding="utf-8")
print(f"📄 Dokument wczytany raz: {len(_PRESIDENTS_RAW)} znaków."
     " Każde zapytanie wyśle go do LLM-a w całości.\n")

def context_stuffing_search(query: str) -> str:
    """Wysyła CAŁY dokument do LLM-a i pyta o odpowiedź."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content":
             "Odpowiadasz WYŁĄCZNIE na podstawie podanych danych o prezydentach. "
             "Jeśli danych brak — powiedz że nie wiesz. Nie wymyślaj. Odpowiadaj po polsku."},
            {"role": "user", "content":
             f"Dane o prezydentach:\n{_PRESIDENTS_RAW}\n\nPytanie: {query}"}
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content

# Przetestujmy na tych samych zapytaniach co substring:
if client:
    _is_local = "127.0.0.1" in str(client.base_url) or "localhost" in str(client.base_url)
    _queries_to_run = test_queries if _is_local else test_queries[:1]

    if _is_local:
        print(f"Context stuffing — lokalny LLM, odpalam {len(_queries_to_run)} zapytań:\n")
    else:
        print(f"⚠️  Serwer prowadzącego — odpalam 1 z {len(test_queries)} zapytań"
              " (żeby nie przeciążać serwera).\n")

    _times = []
    for query, why in _queries_to_run:
        print(f'  "{query}"')
        _t0 = time.time()
        answer = context_stuffing_search(query)
        _dt = time.time() - _t0
        _times.append(_dt)
        print(f"    → {answer[:150]}")
        print(f"    ⏱  {_dt:.1f}s")
        print()

    if _is_local and len(_times) > 1:
        _avg = sum(_times) / len(_times)
        print(f"Średnio {_avg:.1f}s / zapytanie  →"
              f"  {len(test_queries)} pytań = ~{_avg * len(test_queries):.0f}s łącznie.")

    print(f"\nDziała! Ale nasz dokument ma tylko {len(_PRESIDENTS_RAW)} znaków.")
    print("Co jeśli mamy 100 000 znaków? Albo milion? LLM tego nie pomieści.")
else:
    print("LLM niedostępny — przejdź dalej do embeddingów.")

📄 Dokument wczytany raz: 5799 znaków. Każde zapytanie wyśle go do LLM-a w całości.

Context stuffing — lokalny LLM, odpalam 6 zapytań:

  "Nobel"
    → Lech Wałęsa otrzymał Pokojową Nagrodę Nobla w 1983 roku za walkę o prawa pracownicze metodami pokojowymi.
    ⏱  6.2s

  "kto zbierał monety"
    → Andrzej Duda jest zapalonym kolekcjonerem antycznych monet — posiada jedną z największych prywatnych kolekcji monet z czasów Juliusza Cezara w Europie
    ⏱  1.6s

  "najdłużej rządził"
    → Najdłużej prezydenturę w III RP odbył Aleksander Kwaśniewski, którego kadencja trwała 3653 dni.
    ⏱  1.3s

  "związkowiec"
    → Lech Wałęsa był związany z NSZZ „Solidarność”.
    ⏱  0.7s

  "wojskowy"
    → Nie wiem.
    ⏱  0.4s

  "Duda"
    → Oto informacje o prezydencie Andrzeju Dudzie, bazując na podanych danych:

*   **Kadencja:** 2015–2025
*   **Lata życia:** ur. 1972
*   **Miejsce urod
    ⏱  9.7s

Średnio 3.3s / zapytanie  →  6 pytań = ~20s łącznie.

Działa! Ale nasz dokument ma tylko 5799 zn

### Ćwiczenie 1: Porównaj substring vs context stuffing

Uruchom OBA podejścia na własnym pytaniu i porównaj wyniki.
W którym przypadku odpowiedź jest lepsza?

In [7]:
# === Ćwiczenie 1 ===

MOJE_PYTANIE = "kto miał związek z wojskiem?"  # ← ZMIEŃ NA SWOJE

### Proszę poniżej uzupełnić kod ### (≈ 4 linijki)
# Wskazówka: wywołaj search_presidents() i context_stuffing_search() z MOJE_PYTANIE

wynik_substring = search_presidents(MOJE_PYTANIE)  # Tutaj wpisz swój kod
wynik_stuffing = context_stuffing_search(MOJE_PYTANIE)   # Tutaj wpisz swój kod

### Koniec uzupełniania kodu ###

try:
    print(f'Pytanie: "{MOJE_PYTANIE}"\n')
    print(f"Substring search:\n  {wynik_substring[:200]}\n")
    print(f"Context stuffing:\n  {wynik_stuffing[:200]}")
except (TypeError, NameError):
    print("⬆️ Uzupełnij kod powyżej")

Pytanie: "kto miał związek z wojskiem?"

Substring search:
  Nie znaleziono wyników dla zapytania: kto miał związek z wojskiem?

Context stuffing:
  Wśród prezydentów, którzy mieli związki z wojskiem lub służbami mundurowymi, wymienieni są:

*   **Wojciech Jaruzelski:** Był ostatnim przywódcą PRL i wprowadził stan wojenny 13 grudnia 1981 roku. Jeg


###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Użyj `search_presidents(MOJE_PYTANIE)` dla substring search
i `context_stuffing_search(MOJE_PYTANIE)` dla context stuffing.

Pamiętaj o obsłudze przypadku gdy LLM jest niedostępny (`client` jest `None`).

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [34]:
# === Ćwiczenie 1 — rozwiązanie ===

MOJE_PYTANIE = "kto miał związek z wojskiem?"

wynik_substring = search_presidents(MOJE_PYTANIE)
wynik_stuffing = context_stuffing_search(MOJE_PYTANIE) if client else "(LLM niedostępny)"

print(f'Pytanie: "{MOJE_PYTANIE}"\n')
print(f"Substring search:\n  {wynik_substring[:200]}\n")
print(f"Context stuffing:\n  {wynik_stuffing[:200]}")

Pytanie: "kto miał związek z wojskiem?"

Substring search:
  Nie znaleziono wyników dla zapytania: kto miał związek z wojskiem?

Context stuffing:
  Wśród prezydentów, którzy mieli związki z wojskiem lub służbami mundurowymi, wymienieni są:

*   **Wojciech Jaruzelski:** Jego wykształcenie to Wyższa Szkoła Piechoty, a był on również ostatnim przywó


###### 

## 5. Trzy podejścia — porównanie

| | Substring search | Context stuffing | RAG (embeddingi) |
|---|---|---|---|
| **Rozumie synonimy?** | ✗ | ✓ | ✓ |
| **Rozumie kontekst?** | ✗ | ✓ | ✓ |
| **Skaluje się?** | ✓ (szybkie) | ✗ (limit tokenów) | ✓ |
| **Wymaga LLM-a?** | ✗ | ✓ | ✓ (do embeddingów nie) |
| **Koszt** | ~0 | Wysoki (cały dokument) | Niski (tylko top-K) |

Context stuffing to świetne rozwiązanie na małe zbiory danych (do ~10 stron).
Ale gdy mamy **setki lub tysiące** dokumentów — potrzebujemy RAG.

Teraz zbudujemy RAG krok po kroku!

## 6. Przygotowanie dokumentów — chunking

Mamy plik `prezydenci_polski_biografie.txt` z rozbudowanymi biografiami prezydentów Polski.
Zanim zbudujemy RAG, musimy podzielić tekst na **fragmenty (chunks)**.

### Dlaczego dzielimy na fragmenty?
- Modele embeddingowe najlepiej działają na krótszych tekstach (1–3 akapity)
- Chcemy wyszukać **konkretny** fragment, nie cały dokument
- To samo robią profesjonalne systemy RAG (LangChain, LlamaIndex, itd.)

In [35]:
# === Wczytanie dokumentu ===

with open("prezydenci_polski_biografie.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

_md = open("prezydenci_polski.md", encoding="utf-8").read()

print("Porównanie dokumentów:")
print(f"  prezydenci_polski.md        {len(_md):>7,} znaków  ← używaliśmy w context stuffing")
print(f"  prezydenci_polski_biografie {len(full_text):>7,} znaków  "
      f"← {len(full_text)/len(_md):.0f}× więcej")
print()
print(f"Context stuffing z biografiami → {len(full_text):,} znaków przy każdym zapytaniu.")
print(f"Limit modelu: ~128k tokenów ≈ 500k znaków — biografie zajmują {len(full_text)/500_000*100:.1f}% limitu.")
print(f"Wikipedia? Kodeks cywilny? Miliony znaków → context stuffing odpada. Dlatego RAG.")
print()
print(f"Pierwsze 300 znaków biografii:\n")
print(full_text[:300] + "...")


Porównanie dokumentów:
  prezydenci_polski.md          5,799 znaków  ← używaliśmy w context stuffing
  prezydenci_polski_biografie  42,394 znaków  ← 7× więcej

Context stuffing z biografiami → 42,394 znaków przy każdym zapytaniu.
Limit modelu: ~128k tokenów ≈ 500k znaków — biografie zajmują 8.5% limitu.
Wikipedia? Kodeks cywilny? Miliony znaków → context stuffing odpada. Dlatego RAG.

Pierwsze 300 znaków biografii:

PREZYDENCI RZECZYPOSPOLITEJ POLSKIEJ — PEŁNE BIOGRAFIE

Niniejszy dokument zawiera szczegółowe biografie wszystkich prezydentów III Rzeczypospolitej Polskiej,
od Wojciecha Jaruzelskiego (pierwszy prezydent III RP od 31 grudnia 1989) po Karola Na...


In [36]:
# === Podział na fragmenty (chunking) ===

def chunk_text(text, chunk_size=300, overlap=50):
    """
    Dzieli tekst na nakładające się fragmenty.

    Args:
        text: tekst do podziału
        chunk_size: docelowy rozmiar fragmentu (w znakach)
        overlap: ile znaków z końca poprzedniego fragmentu
                 pojawia się na początku następnego

    Dlaczego overlap? Żeby nie "przeciąć" zdania w połowie
    i nie stracić kontekstu na granicy fragmentów.
    """
    # Dzielimy po podwójnych enterach (akapitach)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + 1 + len(para) <= chunk_size:
            current_chunk += (" " if current_chunk else "") + para
        else:
            if current_chunk:
                chunks.append(current_chunk)
                # overlap: nowy chunk zaczyna się od ostatnich N znaków poprzedniego
                tail = current_chunk[-overlap:] if overlap else ""
                current_chunk = (tail + " " + para).strip() if tail else para
            else:
                current_chunk = para

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

chunks = chunk_text(full_text, chunk_size=500, overlap=50)

print(f"Dokument podzielony na {len(chunks)} fragmentów\n")
PREVIEW = 10
for i, chunk in enumerate(chunks[:PREVIEW]):
    print(f"--- Fragment {i+1} ({len(chunk)} znaków) ---")
    print(textwrap.fill(chunk[:150], width=80) + ("..." if len(chunk) > 150 else ""))
    print()

if len(chunks) > PREVIEW:
    print(f"... i jeszcze {len(chunks) - PREVIEW} fragmentów (pokazano {PREVIEW} z {len(chunks)}).")


Dokument podzielony na 117 fragmentów

--- Fragment 1 (414 znaków) ---
PREZYDENCI RZECZYPOSPOLITEJ POLSKIEJ — PEŁNE BIOGRAFIE
====================================================== Niniejszy dokument
zawiera szczegółowe b...

--- Fragment 2 (164 znaków) ---
wanie na podstawie Wikipedii i źródeł publicznych.
======================================== WOJCIECH JARUZELSKI (1989–1990)
==========================...

--- Fragment 3 (477 znaków) ---
989–1990) ======================================== Wojciech Witold Jaruzelski
urodził się 6 lipca 1923 roku w Kurowie, niewielkiej miejscowości położo...

--- Fragment 4 (516 znaków) ---
Miał młodszą siostrę Teresę, urodzoną w 1928 roku. Dzieciństwo i edukacja W
latach 1933–1939 uczęszczał do gimnazjum prowadzonego przez ojców marianów...

--- Fragment 5 (646 znaków) ---
dycji Orląt Lwowskich i bohaterów wojny 1920 roku. Zesłanie na Syberię Po
wybuchu II wojny światowej rodzina Jaruzelskich przekroczyła granicę litewsk...

--- Fragment 6 (465 znaków

### Ćwiczenie 2: Eksperymentuj z rozmiarem fragmentów

Spróbuj zmienić parametry `chunk_size` i `overlap` w komórce powyżej.

> **`overlap`** — ile znaków "zachodzi" na siebie między sąsiednimi fragmentami.
> Np. `overlap=50` przy `chunk_size=300` oznacza, że ostatnie 50 znaków fragmentu N
> pojawia się też na początku fragmentu N+1. Dzięki temu zdanie "przecięte" na granicy
> chunka nie ginie — RAG może je znaleźć z obu stron.

**Proszę uzupełnić poniższą komórkę** — uruchom chunking z innymi parametrami i porównaj wyniki:


In [37]:
# === Ćwiczenie 2 ===
# Spróbuj różnych rozmiarów fragmentów i zobacz jak to wpływa na liczbę chunków

### Proszę poniżej uzupełnić kod ### (≈ 2 linijki)
# Wskazówka: wywołaj chunk_text z innymi parametrami, np. chunk_size=200 lub chunk_size=1000

chunks_male = chunk_text(full_text, chunk_size=200, overlap=30)  # Tutaj wpisz swój kod
chunks_duze = chunk_text(full_text, chunk_size=1000, overlap=100)
# Tutaj wpisz swój kod

### Koniec uzupełniania kodu ###

try:
    print(f"Domyślne (500 znaków): {len(chunks)} fragmentów")
    print(f"Małe (200 znaków):     {len(chunks_male)} fragmentów")
    print(f"Duże (1000 znaków):    {len(chunks_duze)} fragmentów")
    print(f"\nJak myślisz — lepiej mieć więcej małych fragmentów czy mniej dużych? Dlaczego?")
except (TypeError, NameError):
    print("⬆️ Uzupełnij kod powyżej")

Domyślne (500 znaków): 117 fragmentów
Małe (200 znaków):     132 fragmentów
Duże (1000 znaków):    56 fragmentów

Jak myślisz — lepiej mieć więcej małych fragmentów czy mniej dużych? Dlaczego?


###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [38]:
# === Ćwiczenie 2 — rozwiązanie ===

chunks_male = chunk_text(full_text, chunk_size=200, overlap=30)
chunks_duze = chunk_text(full_text, chunk_size=1000, overlap=100)

print(f"Domyślne (500 znaków): {len(chunks)} fragmentów")
print(f"Małe (200 znaków):     {len(chunks_male)} fragmentów")
print(f"Duże (1000 znaków):    {len(chunks_duze)} fragmentów")

print(f"\nPrzykładowy mały fragment:")
print(f"  '{chunks_male[0][:100]}...'")
print(f"\nPrzykładowy duży fragment:")
print(f"  '{chunks_duze[0][:100]}...'")

Domyślne (500 znaków): 117 fragmentów
Małe (200 znaków):     132 fragmentów
Duże (1000 znaków):    56 fragmentów

Przykładowy mały fragment:
  'PREZYDENCI RZECZYPOSPOLITEJ POLSKIEJ — PEŁNE BIOGRAFIE
=============================================...'

Przykładowy duży fragment:
  'PREZYDENCI RZECZYPOSPOLITEJ POLSKIEJ — PEŁNE BIOGRAFIE
=============================================...'


###### 

## 7. Embeddingi — zamiana tekstu na wektory

Pamiętacie **Word2Vec** z poprzednich zajęć? Tam każde *słowo* miało swój wektor.
Teraz robimy to samo, ale dla **całych fragmentów tekstu** — to są tzw. *sentence embeddings*.

Używamy modelu `paraphrase-multilingual-MiniLM-L12-v2` — lekkiego modelu (do uruchomienia na CPU),
który rozumie wiele języków, w tym **polski**.

### Jak to działa?
```
"Prezydent Kaczyński zginął w katastrofie" → [0.12, -0.45, 0.78, ..., 0.03]  (384 liczby)
```

Teksty o podobnym *znaczeniu* będą miały **bliskie wektory** (mały kąt między nimi = wysokie cosine similarity).

In [39]:
# === Ładowanie modelu embeddingowego ===
# Ten model działa na CPU — nie potrzeba GPU!
# Przy pierwszym uruchomieniu zostanie pobrany (~120 MB)

print("Ładowanie modelu embeddingowego...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print(f"Model załadowany!")
print(f"Wymiar embeddingów: {embed_model.get_sentence_embedding_dimension()}")

Ładowanie modelu embeddingowego...
Model załadowany!
Wymiar embeddingów: 384


In [40]:
# === Tworzenie embeddingów dla naszych fragmentów ===

print(f"Tworzę embeddingi dla {len(chunks)} fragmentów...")
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)

print(f"\nGotowe! Kształt macierzy embeddingów: {chunk_embeddings.shape}")
print(f"Każdy fragment to wektor {chunk_embeddings.shape[1]} liczb\n")

# Zobaczmy jak wygląda embedding pierwszego fragmentu
print(f"Embedding fragmentu 1 (pierwsze 10 wartości):")
print(np.round(chunk_embeddings[0][:10], 4))

Tworzę embeddingi dla 117 fragmentów...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Gotowe! Kształt macierzy embeddingów: (117, 384)
Każdy fragment to wektor 384 liczb

Embedding fragmentu 1 (pierwsze 10 wartości):
[-0.3313  0.3363 -0.0508 -0.1727  0.0801  0.1307 -0.0415  0.2061  0.0956
  0.1349]


## 8. Cosine similarity — miara podobieństwa wektorów

### Ćwiczenie 3: Cosine similarity "ręcznie"

Zanim użyjemy gotowej funkcji wyszukiwania, policzmy cosine similarity sami.

Przypomnijmy wzór:

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

Gdzie $A \cdot B$ to iloczyn skalarny, a $\|A\|$ to norma (długość) wektora.

In [41]:
# === Ćwiczenie 3: Policz cosine similarity ręcznie ===

try:
    # Pytanie o Nagrodę Nobla — porównajmy z dwoma fragmentami:
    #   chunks[23] → o Wałęsie i Pokojowej Nagrodzie Nobla (powinien być WYSOKI)
    #   chunks[5]  → o służbie wojskowej Jaruzelskiego (powinien być NISKI)
    pytanie = "Kto otrzymał Pokojową Nagrodę Nobla?"
    emb_pytania = embed_model.encode([pytanie])[0]
    emb_fragment_a = chunk_embeddings[23]   # Wałęsa — Nobel
    emb_fragment_b = chunk_embeddings[5]    # Jaruzelski — wojsko

    ### Proszę poniżej uzupełnić kod ### (≈ 2 linijki)
    # Wskazówka: użyj np.dot() do iloczynu skalarnego i np.linalg.norm() do normy

    similarity_a = ...  # Tutaj wpisz swój kod
    similarity_b = ...  # Tutaj wpisz swój kod

    ### Koniec uzupełniania kodu ###

    print(f"Pytanie: '{pytanie}'\n")
    print(f"Fragment A (Wałęsa — Nobel):")
    print(f"  '{chunks[23][:100]}...'")
    print(f"  Cosine similarity: {similarity_a:.4f}\n")
    print(f"Fragment B (Jaruzelski — wojsko):")
    print(f"  '{chunks[5][:100]}...'")
    print(f"  Cosine similarity: {similarity_b:.4f}\n")
    print(f"Który fragment jest bliższy pytaniu? {'Fragment A ✓' if similarity_a > similarity_b else 'Fragment B'}")
except (TypeError, NameError):
    print("⬆️ Uzupełnij kod powyżej")

Pytanie: 'Kto otrzymał Pokojową Nagrodę Nobla?'

Fragment A (Wałęsa — Nobel):
  'ąc kontakt z podziemnymi strukturami
Solidarności. Pokojowa Nagroda Nobla
5 października 1983 roku K...'
⬆️ Uzupełnij kod powyżej


###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Cosine similarity to iloczyn skalarny podzielony przez iloczyn norm:

$$\text{sim}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

W NumPy: `np.dot(a, b)` daje iloczyn skalarny, `np.linalg.norm(a)` daje normę wektora.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [42]:
# === Ćwiczenie 3 — rozwiązanie ===

pytanie = "Kto otrzymał Pokojową Nagrodę Nobla?"
emb_pytania = embed_model.encode([pytanie])[0]
emb_fragment_a = chunk_embeddings[23]   # Wałęsa — Nobel
emb_fragment_b = chunk_embeddings[5]    # Jaruzelski — wojsko

similarity_a = np.dot(emb_pytania, emb_fragment_a) / (np.linalg.norm(emb_pytania) * np.linalg.norm(emb_fragment_a))
similarity_b = np.dot(emb_pytania, emb_fragment_b) / (np.linalg.norm(emb_pytania) * np.linalg.norm(emb_fragment_b))

print(f"Pytanie: '{pytanie}'\n")
print(f"Fragment A (Wałęsa — Nobel):")
print(f"  '{chunks[23][:100]}...'")
print(f"  Cosine similarity: {similarity_a:.4f}\n")
print(f"Fragment B (Jaruzelski — wojsko):")
print(f"  '{chunks[5][:100]}...'")
print(f"  Cosine similarity: {similarity_b:.4f}\n")
print(f"Który fragment jest bliższy pytaniu? {'Fragment A ✓' if similarity_a > similarity_b else 'Fragment B'}")
print(f"\nRóżnica: {abs(similarity_a - similarity_b):.4f} — embeddingi wyraźnie 'wiedzą' który tekst pasuje!")

Pytanie: 'Kto otrzymał Pokojową Nagrodę Nobla?'

Fragment A (Wałęsa — Nobel):
  'ąc kontakt z podziemnymi strukturami
Solidarności. Pokojowa Nagroda Nobla
5 października 1983 roku K...'
  Cosine similarity: 0.6916

Fragment B (Jaruzelski — wojsko):
  'oletni Wojciech został jedynym żywicielem rodziny. Służba wojskowa w czasie II wojny światowej
19 li...'
  Cosine similarity: 0.2199

Który fragment jest bliższy pytaniu? Fragment A ✓

Różnica: 0.4716 — embeddingi wyraźnie 'wiedzą' który tekst pasuje!


###### 

## 9. Wyszukiwanie semantyczne

Teraz najciekawsza część! Zamieniamy pytanie użytkownika na embedding,
a potem szukamy **najbliższych** fragmentów tekstu.

"Najbliższy" = najwyższe **cosine similarity** (podobieństwo kosinusowe).

To jest *serce* RAG-a — wyszukiwanie semantyczne. Zauważcie, że nie szukamy po słowach kluczowych
(jak w Google), ale po **znaczeniu** tekstu!

In [43]:
# === Funkcja wyszukiwania semantycznego ===

def search(query, chunks, chunk_embeddings, embed_model, top_k=3):
    """
    Wyszukuje top_k najbardziej pasujących fragmentów do zapytania.
    
    Kroki:
    1. Zamień pytanie na embedding
    2. Oblicz cosine similarity z każdym fragmentem
    3. Zwróć top_k najlepszych
    """
    # 1. Embedding pytania
    query_embedding = embed_model.encode([query])[0]
    
    # 2. Cosine similarity = dot product znormalizowanych wektorów
    #    similarity(A, B) = (A · B) / (||A|| * ||B||)
    similarities = []
    for i, chunk_emb in enumerate(chunk_embeddings):
        dot_product = np.dot(query_embedding, chunk_emb)
        norm_q = np.linalg.norm(query_embedding)
        norm_c = np.linalg.norm(chunk_emb)
        similarity = dot_product / (norm_q * norm_c)
        similarities.append((i, similarity, chunks[i]))
    
    # 3. Sortuj malejąco po similarity
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]

print("Funkcja wyszukiwania gotowa!")

Funkcja wyszukiwania gotowa!


In [44]:
# === Testujemy wyszukiwanie! ===

query = "Kto i za co otrzymał Pokojową Nagrodę Nobla?"

print(f"Pytanie: {query}")
print("="*70)

results = search(query, chunks, chunk_embeddings, embed_model, top_k=3)

print(f"\nZnaleziono {len(results)} najbardziej trafnych fragmentów:\n")

for rank, (idx, score, chunk_text) in enumerate(results, 1):
    print(f"{'─'*70}")
    print(f"  Trafność #{rank} | Fragment nr {idx+1} | Cosine similarity: {score:.4f}")
    print(f"{'─'*70}")
    # Pokazujemy cały fragment — student widzi CO DOKŁADNIE embeddingi znalazły
    print(textwrap.fill(chunk_text, width=70, initial_indent="  ", subsequent_indent="  "))
    print()

Pytanie: Kto i za co otrzymał Pokojową Nagrodę Nobla?

Znaleziono 3 najbardziej trafnych fragmentów:

──────────────────────────────────────────────────────────────────────
  Trafność #1 | Fragment nr 24 | Cosine similarity: 0.6942
──────────────────────────────────────────────────────────────────────
  ąc kontakt z podziemnymi strukturami Solidarności. Pokojowa Nagroda
  Nobla 5 października 1983 roku Komitet Noblowski przyznał Wałęsie
  Pokojową Nagrodę Nobla za „kluczowy wkład w potwierdzenie wolności
  zrzeszania się we wszystkich krajach". W uzasadnieniu podkreślono
  jego „odważny wybór drogi pokoju". Wałęsa nie mógł pojechać do Oslo
  — obawiał się, że nie zostanie wpuszczony z powrotem do Polski.
  Nagrodę w jego imieniu odebrała żona Danuta wraz z trzynastoletnim
  synem Bogdanem. Mowę noblowską odczytał Bohdan Cywiński.

──────────────────────────────────────────────────────────────────────
  Trafność #2 | Fragment nr 114 | Cosine similarity: 0.6574
──────────────────────────

In [25]:
# === Sprawdźmy inne pytania! ===

semantic_test_queries = [
    "Kto był najmłodszym prezydentem Polski?",
    "Który prezydent wprowadził stan wojenny i kiedy?",
    "Jak długo trwała kadencja Aleksandra Kwaśniewskiego?",
]

for q in semantic_test_queries:
    print(f"\n{'='*70}")
    print(f"PYTANIE: {q}")
    print(f"{'='*70}")
    results = search(q, chunks, chunk_embeddings, embed_model, top_k=2)
    for rank, (idx, score, text) in enumerate(results, 1):
        print(f"\n  #{rank} [similarity: {score:.4f}] Fragment {idx+1}:")
        print(textwrap.fill(text[:200], width=70, initial_indent="  ", subsequent_indent="  "))
        if len(text) > 200:
            print("  ...")


PYTANIE: Kto był najmłodszym prezydentem Polski?

  #1 [similarity: 0.6624] Fragment 40:
  wyższy indywidualny wynik w kraju — 148 553 głosy. Pierwsza kadencja
  (1995–2000) W wyborach prezydenckich 1995 roku, startując pod hasłem
  „Wybierzmy przyszłość", Kwaśniewski pokonał w drugiej turze ur
  ...

  #2 [similarity: 0.6591] Fragment 4:
  Miał młodszą siostrę Teresę, urodzoną w 1928 roku. Dzieciństwo i
  edukacja W latach 1933–1939 uczęszczał do gimnazjum prowadzonego
  przez ojców marianów na warszawskich Bielanach. Szkoła kładła nacisk
  n
  ...

PYTANIE: Którzy prezydenci zginęli w trakcie sprawowania urzędu?

  #1 [similarity: 0.5622] Fragment 115:
  agrodę odebrała jego żona Danuta z synem Bogdanem. Drogi do
  prezydentury Każdy z prezydentów III RP dotarł na urząd inną
  ścieżką: — Jaruzelski: generał, I Sekretarz KC PZPR, premier →
  wybrany przez Zg
  ...

  #2 [similarity: 0.5612] Fragment 1:
  PREZYDENCI RZECZYPOSPOLITEJ POLSKIEJ — PEŁNE BIOGRAFIE
  =======================

## 10. RAG — łączymy wyszukiwanie z LLM-em!

Teraz kluczowy moment. Mamy:
- **Pytanie** użytkownika
- **Fragmenty** tekstu znalezione przez wyszukiwanie semantyczne

Składamy to razem w **prompt** dla LLM-a:

```
Kontekst (znalezione fragmenty):
[Fragment 1]
[Fragment 2]
[Fragment 3]

Pytanie: ...
Odpowiedz WYŁĄCZNIE na podstawie powyższego kontekstu.
```

To jest cały "sekret" RAG-a — dajemy LLM-owi kontekst, którego sam nie ma!

In [45]:
# === Budowanie prompta RAG ===

TEST_QUESTION = "Jak długo trwała kadencja Aleksandra Kwaśniewskiego i ile miał lat obejmując urząd?"

def build_rag_prompt(query, retrieved_chunks):
    """
    Buduje prompt RAG z pytaniem i znalezionymi fragmentami.
    """
    context_parts = []
    for rank, (idx, score, text) in enumerate(retrieved_chunks, 1):
        context_parts.append(f"[Fragment {rank} | trafność: {score:.3f}]\n{text}")
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""Na podstawie poniższych fragmentów odpowiedz na pytanie.
Używaj WYŁĄCZNIE informacji z podanych fragmentów. Nie wymyślaj faktów.
Jeśli fragmenty nie zawierają odpowiedzi, napisz dokładnie:
"Brak informacji w dostarczonych fragmentach."

=== FRAGMENTY ===
{context}
=== KONIEC FRAGMENTÓW ===

Pytanie: {query}

Odpowiedź:"""
    
    return prompt

# Przetestujmy jak wygląda gotowy prompt
results = search(TEST_QUESTION, chunks, chunk_embeddings, embed_model, top_k=3)
rag_prompt = build_rag_prompt(TEST_QUESTION, results)

print("=== GOTOWY PROMPT RAG (to idzie do LLM-a) ===")
print(rag_prompt)
print(f"\n\nDługość prompta: {len(rag_prompt)} znaków")

=== GOTOWY PROMPT RAG (to idzie do LLM-a) ===
Na podstawie poniższych fragmentów odpowiedz na pytanie.
Używaj WYŁĄCZNIE informacji z podanych fragmentów. Nie wymyślaj faktów.
Jeśli fragmenty nie zawierają odpowiedzi, napisz dokładnie:
"Brak informacji w dostarczonych fragmentach."

=== FRAGMENTY ===
[Fragment 1 | trafność: 0.755]
EKAWOSTKI
======================================== Długość kadencji prezydentów III RP
— Aleksander Kwaśniewski: 3653 dni (najdłuższa — dwie pełne kadencje)
— Andrzej Duda: 3653 dni (dwie pełne kadencje, tyle samo co Kwaśniewski)
— Lech Wałęsa: 1826 dni (jedna kadencja)
— Bronisław Komorowski: 1826 dni (jedna kadencja)
— Lech Kaczyński: 1631 dni (kadencja przerwana katastrofą smoleńską)
— Karol Nawrocki: kadencja trwa (od sierpnia 2025)
— Wojciech Jaruzelski: 356 dni (najkrótsza w III RP)

[Fragment 2 | trafność: 0.741]
Wojciech Jaruzelski: 356 dni (najkrótsza w III RP) Wiek w dniu objęcia urzędu
— Aleksander Kwaśniewski: 41 lat (najmłodszy)
— Karol Nawrocki: 

In [48]:
# === Odpowiedź LLM-a Z RAG-iem ===

# Budujemy prompt od nowa (nie polegamy na zmiennej z innej komórki)
results = search(TEST_QUESTION, chunks, chunk_embeddings, embed_model, top_k=3)
rag_prompt = build_rag_prompt(TEST_QUESTION, results)

if client:
    print(f"Pytanie: {TEST_QUESTION}")
    print(f"Model: {MODEL_NAME}")
    print("="*60)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku, krótko i na temat."},
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
    )
    
    rag_answer = response.choices[0].message.content
    print(f"\nOdpowiedź LLM-a Z RAG-iem:\n")
    print(textwrap.fill(rag_answer, width=80))
else:
    print("LLM niedostępny. Powyżej widać prompt który normalnie poszedłby do modelu.")

Pytanie: Jak długo trwała kadencja Aleksandra Kwaśniewskiego i ile miał lat obejmując urząd?
Model: gemma-4-e4b-it-mlx

Odpowiedź LLM-a Z RAG-iem:

Kadencja Aleksandra Kwaśniewskiego trwała 3653 dni, a objął urząd w wieku 41
lat.


## 11. Porównanie: BEZ RAG vs Z RAG

Zobaczmy różnicę na wielu pytaniach!

In [49]:
# === Pełne porównanie: bez RAG vs z RAG ===

questions = [
    "Kto był najmłodszym prezydentem Polski i ile miał lat obejmując urząd?",
    "Który prezydent zginął w katastrofie lotniczej i kiedy to się wydarzyło?",
    "Kto i za co otrzymał Pokojową Nagrodę Nobla?",
    # ↓ Pytanie-pułapka: Zaleski był prezydentem na uchodźstwie, nie III RP
    #   RAG powinien odpowiedzieć "Brak informacji" — i to jest POPRAWNA odpowiedź!
    "Jak długo August Zaleski pełnił funkcję prezydenta?",
]

if client:
    for q in questions:
        print(f"\n{'='*70}")
        print(f"PYTANIE: {q}")
        print(f"{'='*70}")
        
        # Bez RAG
        resp_no_rag = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Odpowiadaj krótko i po polsku."},
                {"role": "user", "content": q}
            ],
            temperature=0.1,
        )
        
        # Z RAG
        results = search(q, chunks, chunk_embeddings, embed_model, top_k=3)
        rag_prompt = build_rag_prompt(q, results)
        
        resp_rag = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku, krótko i na temat."},
                {"role": "user", "content": rag_prompt}
            ],
            temperature=0.1,
        )
        
        print(f"\n  BEZ RAG: {resp_no_rag.choices[0].message.content[:200]}")
        print(f"\n  Z RAG:   {resp_rag.choices[0].message.content[:200]}")
        
        # Pokaż użyte fragmenty
        print(f"\n  Użyte fragmenty (top {len(results)}):")
        for rank, (idx, score, text) in enumerate(results, 1):
            print(f"    #{rank} [sim={score:.3f}]: {text[:80]}...")
else:
    print("LLM niedostępny — porównanie wymaga modelu.")


PYTANIE: Kto był najmłodszym prezydentem Polski i ile miał lat obejmując urząd?

  BEZ RAG: Najmłodszym prezydentem Polski był Lech Kaczyński, który objął urząd w wieku 51 lat.

  Z RAG:   Aleksander Kwaśniewski był najmłodszym prezydentem III RP i miał 41 lat w dniu objęcia urzędu.

  Użyte fragmenty (top 3):
    #1 [sim=0.692]: wyższy indywidualny wynik w kraju — 148 553 głosy. Pierwsza kadencja (1995–2000)...
    #2 [sim=0.687]: wielką przewagą. Objął
urząd 6 sierpnia 2025 roku. Jest drugim najmłodszym prezy...
    #3 [sim=0.650]: Wojciech Jaruzelski: 356 dni (najkrótsza w III RP) Wiek w dniu objęcia urzędu
— ...

PYTANIE: Który prezydent zginął w katastrofie lotniczej i kiedy to się wydarzyło?

  BEZ RAG: Nie podaję informacji o konkretnym prezydencie, który zginął w katastrofie lotniczej, ponieważ mogłoby to być zbyt ogólne lub nieprecyzyjne.

  Z RAG:   Prezydent Kaczyński zginął w katastrofie smoleńskiej, która miała miejsce 10 kwietnia 2010 roku.

  Użyte fragmenty (top 3):
  

## 12. Zadaj własne pytanie!

Teraz Twoja kolej — wpisz pytanie o prezydentach Polski i zobacz jak działa RAG.

In [50]:
# === Twoje pytanie! ===

TWOJE_PYTANIE = "W jakim mieście Lech Wałęsa pracował w stoczni?"  # <-- ZMIEŃ NA SWOJE!


# --- Wyszukiwanie semantyczne ---
print(f"Pytanie: {TWOJE_PYTANIE}")
print("="*70)

results = search(TWOJE_PYTANIE, chunks, chunk_embeddings, embed_model, top_k=3)

print(f"\nZnalezione fragmenty:\n")
for rank, (idx, score, text) in enumerate(results, 1):
    print(f"  #{rank} [cosine similarity: {score:.4f}]:")
    print(textwrap.fill(text, width=70, initial_indent="    ", subsequent_indent="    "))
    print()

# --- RAG ---
if client:
    rag_prompt = build_rag_prompt(TWOJE_PYTANIE, results)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku."},
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
    )
    
    print(f"{'='*70}")
    print(f"Odpowiedź RAG:\n")
    print(textwrap.fill(response.choices[0].message.content, width=70))
else:
    print("LLM niedostępny — ale powyżej widzisz co wyszukiwanie znalazło!")

Pytanie: Ile państw uznawało rząd RP na uchodźstwie?

Znalezione fragmenty:

  #1 [cosine similarity: 0.4469]:
    jąc 55,2% głosów na I Krajowym Zjeździe Delegatów. Stan wojenny i
    internowanie 13 grudnia 1981 roku, po wprowadzeniu stanu wojennego
    przez generała Jaruzelskiego, Wałęsa został internowany. Władze
    liczyły, że uda się go wykorzystać jako figuranta kontrolowanej
    „Solidarności", ale odmówił negocjacji. Przetrzymywany był kolejno
    w Chylicach, Otwocku Wielkim i Arłamowie. Formalnie internowanie
    datowane jest od 26 stycznia 1982 roku. Zwolniony został 14
    listopada 1982 roku dzięki staraniom generała Czesława Kiszczaka.
    Przez resztę lat 80. pozostawał pod inwigilacją Służby
    Bezpieczeństwa, jednocześnie utrzymując kontakt z podziemnymi
    strukturami Solidarności.

  #2 [cosine similarity: 0.3777]:
    Senacie i wszystkie 161 wolnych mandatów w Sejmie. W sierpniu 1989
    roku Wałęsa doprowadził do koalicji z Zjednoczonym Stronnictwem
    Ludow

## 13. Co pod spodem? Wizualizacja embeddingów

Zobaczmy **jak** wyszukiwanie semantyczne wybiera fragmenty.
Trzy wykresy poniżej pokazują to z różnych perspektyw:

1. **Ranking podobieństwa** — jak bardzo każdy fragment pasuje do pytania (i które trafiają do RAG)
2. **Porównanie dwóch pytań** — inne pytanie = inne fragmenty na szczycie
3. **Rozkład podobieństwa** — większość fragmentów jest daleko, tylko kilka jest blisko (igła w stogu siana)

In [ ]:
# === Wizualizacja embeddingów — 3 wykresy ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

TOP_K = 3  # ile fragmentów trafia do RAG

# ---------- WYKRES 1: Ranking podobieństwa ----------
# "Jak bardzo każdy fragment pasuje do pytania?"

query1 = "Kto i za co otrzymał Pokojową Nagrodę Nobla?"
results_all = search(query1, chunks, chunk_embeddings, embed_model, top_k=len(chunks))
indices = [idx for idx, _, _ in results_all]
scores  = [score for _, score, _ in results_all]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#e74c3c' if i < TOP_K else '#3498db' for i in range(len(scores))]
bars = ax.bar(range(len(scores)), scores, color=colors, width=0.8, alpha=0.85)

# Etykiety top-K
for i in range(TOP_K):
    ax.text(i, scores[i] + 0.01, f"F{indices[i]+1}\n{scores[i]:.3f}",
            ha='center', va='bottom', fontsize=8, fontweight='bold', color='#c0392b')

ax.set_xlabel("Fragmenty (posortowane od najbardziej do najmniej trafnych)", fontsize=11)
ax.set_ylabel("Cosine similarity", fontsize=11)
ax.set_title(f"Ranking podobieństwa do pytania:\n\"{query1}\"", fontsize=13)
ax.axhline(y=scores[TOP_K-1], color='red', linestyle='--', alpha=0.5, label=f'Próg top-{TOP_K}')
ax.set_xticks([])
red_patch = mpatches.Patch(color='#e74c3c', label=f'Top {TOP_K} → trafiają do RAG')
blue_patch = mpatches.Patch(color='#3498db', label='Pozostałe fragmenty')
ax.legend(handles=[red_patch, blue_patch], loc='upper right', fontsize=10)
ax.set_ylim(0, max(scores) * 1.15)
plt.tight_layout()
plt.show()

print(f"Tylko {TOP_K} z {len(chunks)} fragmentów trafia do prompta RAG.")
print(f"Top-1 (similarity {scores[0]:.3f}) vs ostatni ({scores[-1]:.3f}) — różnica {scores[0]-scores[-1]:.3f}\n")

# ---------- WYKRES 2: Dwa pytania — inne fragmenty ----------
# "Inne pytanie → inne fragmenty na szczycie!"

query_a = "Kto i za co otrzymał Pokojową Nagrodę Nobla?"
query_b = "Który prezydent zginął w katastrofie lotniczej?"

results_a = search(query_a, chunks, chunk_embeddings, embed_model, top_k=len(chunks))
results_b = search(query_b, chunks, chunk_embeddings, embed_model, top_k=len(chunks))

# Budujemy macierz: dla każdego fragmentu → similarity do obu pytań
sim_a = {idx: score for idx, score, _ in results_a}
sim_b = {idx: score for idx, score, _ in results_b}
all_indices = sorted(set(sim_a.keys()))

# Sortujemy po sim_a (pytanie A)
vals_a = [sim_a[i] for i in all_indices]
vals_b = [sim_b[i] for i in all_indices]

# Top-K dla każdego pytania
top_a = set(idx for idx, _, _ in results_a[:TOP_K])
top_b = set(idx for idx, _, _ in results_b[:TOP_K])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Pytanie A
colors_a = ['#e74c3c' if i in top_a else '#bdc3c7' for i in all_indices]
ax1.bar(range(len(all_indices)), vals_a, color=colors_a, width=0.8, alpha=0.85)
ax1.set_ylabel("Cosine similarity", fontsize=10)
ax1.set_title(f"Pytanie A: \"{query_a}\"", fontsize=11, color='#e74c3c')
ax1.set_ylim(0, max(max(vals_a), max(vals_b)) * 1.15)

# Pytanie B
colors_b = ['#2ecc71' if i in top_b else '#bdc3c7' for i in all_indices]
ax2.bar(range(len(all_indices)), vals_b, color=colors_b, width=0.8, alpha=0.85)
ax2.set_ylabel("Cosine similarity", fontsize=10)
ax2.set_title(f"Pytanie B: \"{query_b}\"", fontsize=11, color='#27ae60')
ax2.set_xlabel(f"Fragmenty dokumentu (1–{len(chunks)})", fontsize=10)
ax2.set_ylim(0, max(max(vals_a), max(vals_b)) * 1.15)

# Etykiety top-K
for ax, results, color in [(ax1, results_a, '#c0392b'), (ax2, results_b, '#1e8449')]:
    for rank, (idx, score, _) in enumerate(results[:TOP_K]):
        pos = all_indices.index(idx)
        ax.text(pos, score + 0.01, f"F{idx+1}", ha='center', fontsize=7,
                fontweight='bold', color=color)

fig.suptitle("Inne pytanie → inne fragmenty trafiają do RAG", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Pytanie A wybiera fragmenty: {sorted([idx+1 for idx in top_a])}")
print(f"Pytanie B wybiera fragmenty: {sorted([idx+1 for idx in top_b])}")
overlap = top_a & top_b
print(f"Wspólne: {sorted([idx+1 for idx in overlap]) if overlap else 'BRAK'} — każde pytanie trafia w inne miejsce!\n")

# ---------- WYKRES 3: Histogram — igła w stogu siana ----------
# "Większość fragmentów jest daleko, tylko kilka blisko"

fig, ax = plt.subplots(figsize=(10, 5))

all_scores = [score for _, score, _ in results_a]
ax.hist(all_scores, bins=25, color='#3498db', alpha=0.7, edgecolor='white', linewidth=0.5)

# Zaznacz próg top-K
threshold = sorted(all_scores, reverse=True)[TOP_K-1]
ax.axvline(x=threshold, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Próg top-{TOP_K} (sim={threshold:.3f})')

# Zaznacz obszar top-K
ax.axvspan(threshold, max(all_scores) * 1.05, alpha=0.15, color='#e74c3c')
ax.text(threshold + 0.01, ax.get_ylim()[1] * 0.85,
        f"← Te {TOP_K} fragmenty\n   trafiają do RAG",
        fontsize=10, color='#c0392b', fontweight='bold')

ax.set_xlabel("Cosine similarity do pytania", fontsize=11)
ax.set_ylabel("Liczba fragmentów", fontsize=11)
ax.set_title(f"Rozkład podobieństwa: \"{query_a[:50]}...\"\n"
             f"Szukamy igły w stogu siana — {TOP_K} z {len(chunks)} fragmentów", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Średnie similarity: {np.mean(all_scores):.3f}")
print(f"Top-{TOP_K} similarity: {', '.join(f'{s:.3f}' for s in sorted(all_scores, reverse=True)[:TOP_K])}")
print(f"Większość fragmentów ma similarity < {np.median(all_scores):.3f} (mediana)")
print(f"\nTo jest serce RAG-a: z {len(chunks)} fragmentów wybieramy {TOP_K} najbardziej trafne!")

## Podsumowanie

### Co zrobiliśmy?

1. **Substring search** — zobaczyliśmy że wyszukiwanie po słowach kluczowych nie rozumie języka
2. **Context stuffing** — wrzuciliśmy cały dokument do LLM-a → działa, ale nie skaluje się
3. **Chunking** — podzieliliśmy tekst na fragmenty
4. **Embeddingi** — zamieniliśmy fragmenty na wektory liczbowe (384 wymiary)
5. **Wyszukiwanie semantyczne** — znaleźliśmy fragmenty najbliższe pytaniu (cosine similarity)
6. **RAG** — połączyliśmy wyszukiwanie z LLM-em → poprawne odpowiedzi!

### Trzy podejścia — kiedy co?

| | Substring | Context stuffing | RAG |
|---|---|---|---|
| **Kiedy?** | Mały, strukturalny zbiór | < 10 stron tekstu | Duże zbiory dokumentów |
| **Przykład** | `search_presidents()` z FC | Cały .md do prompta | Embeddingi + top-K + LLM |

### RAG w produkcji

W prawdziwych zastosowaniach zamiast naszego prostego kodu używa się:
- **Wektorowych baz danych** (Pinecone, Weaviate, ChromaDB, FAISS) zamiast numpy
- **Frameworków** (LangChain, LlamaIndex) do orkiestracji
- **Lepszych strategii chunkingu** (recursive, semantic chunking)
- **Re-rankingu** (dodatkowy model który ponownie ocenia trafność)
- **Hybrydowego wyszukiwania** (embeddingi + klasyczny BM25)
- **Multi-query RAG** (LLM generuje kilka wariantów pytania → lepsze pokrycie)

Ale **zasada jest dokładnie taka sama** jak to co zrobiliśmy na tych zajęciach!

### Kiedy RAG, a kiedy fine-tuning?

| | RAG | Fine-tuning (SFT) |
|---|---|---|
| **Kiedy?** | Dane się zmieniają, trzeba cytować źródła | Chcemy zmienić "styl" lub "wiedzę" modelu |
| **Koszt** | Tani (tylko inference) | Drogi (trening na GPU) |
| **Aktualność** | Zawsze aktualne (aktualizujesz dokumenty) | Wymaga ponownego treningu |
| **Halucynacje** | Mniejsze (model ma kontekst) | Możliwe (model "pamięta" błędnie) |